# Detector Training - YOLO26 on PRW
| ID  | Student       | Email                  |
|-----|---------------|------------------------|
| 0001189110   | Rimondi Simone   | simone.rimondi5@studio.unibo.it|

## 1. Introduction

This notebook presents a structured ablation study aimed at assessing the transferability of **YOLO26**, the 2026 iteration of the YOLO family of real-time object detectors, to the task of pedestrian detection on the **PRW** (*Person Re-identification in the Wild*) benchmark. The study is conducted across two model scales (Small and Large) and evaluates the impact of domain-specific intermediate pre-training and fine-tuning strategy on detection performance.

### Context: Two-Stage Person Search

This detection module constitutes the first stage of a **two-stage person search pipeline**. In this paradigm, a detector is responsible for localising all pedestrian instances in a scene, and a subsequent re-identification (Re-ID) module matches each detected crop against a query identity. Since any missed detection or poor localisation at this stage propagates irrecoverably to the Re-ID step, the quality of the detector is a critical bottleneck for the overall system performance. Maximising recall while maintaining high precision is therefore a primary design objective of this ablation.

### Motivation for YOLO

YOLO was selected as the detector backbone for three main reasons. First, it represents the current state of the art in real-time object detection, offering an excellent trade-off between accuracy and inference speed. Second, the availability of officially released pre-trained weights (on COCO) enables robust transfer learning even with limited target-domain data. Third, the `ultralytics` library provides a clean, well-documented API that significantly reduces implementation overhead and facilitates rapid experimentation across model scales and fine-tuning strategies.

### Objective

The primary research question is: *to what extent does a two-stage transfer learning pipeline, first adapting the model on a large crowd-scene dataset and then fine-tuning on the target domain, improve over a direct zero-shot or single-stage fine-tuning baseline?* A secondary axis of investigation concerns the choice of fine-tuning strategy: **full fine-tuning** (all parameters updated) versus **partial fine-tuning** (backbone frozen, only neck and detection head trained), which directly probes the generalization capacity of the learned feature representations.

### Datasets

- **PRW** (Person Re-identification in the Wild): a surveillance-domain dataset consisting of video frames captured by static cameras in a university campus. It contains 11,816 frames, 932 labelled identities, and 34,304 bounding-box annotations for the *person* class. Annotations are provided in MATLAB format and converted to YOLO-compatible normalised coordinates (`class x_c y_c w h`) as a preprocessing step.
- **CrowdHuman** (`leducnhuan/crowdhuman`): a large-scale benchmark specifically designed for person detection in crowded scenarios (15k training images, 4.4k validation images). It was selected as the intermediate pre-training domain because its density and occlusion statistics are closer to the surveillance conditions of PRW than alternative datasets such as MOT or WIDER PERSON, making it a more effective bridge domain for feature adaptation before target fine-tuning.

### Methodology

The experimental pipeline is organised into five sequential phases, summarised in the table below. For each training run the same standardised hyperparameter set is applied (AdamW optimiser, cosine learning-rate decay with linear warm-up, standard YOLO augmentation suite including mosaic, mixup, and horizontal flip). Evaluation is performed at fixed confidence threshold and IoU threshold (0.5), and each model is assessed on four metrics: **mAP@0.5**, **mAP@0.5:0.95**, **Precision**, and **Recall**. Profiling functions adds parameter count, GFLOPs, per-image latency, and throughput, enabling a quality-vs-compute trade-off analysis.

| Phase | Description |
|---|---|
| **0 - Baseline** | YOLO26-Small COCO zero-shot eval on PRW |
| **1 - CrowdHuman Pre-training** | Fine-tune Small on CrowdHuman, then eval on PRW |
| **2 - Strategy Comparison** | Full FT vs Partial FT |
| **3 - Scale-up** | Best strategy on YOLO26-Large |
| **Final** | Cross-model comparison: metrics, params, FLOPs, speed |

### Reproducibility

All experiments use a fixed global seed (42) applied to Python, NumPy, and PyTorch. Results are incrementally persisted to a JSON registry (`all_results.json`) and exported as a CSV table at the end of the notebook. It is possible to regenerate all figures after a kernel restart without repeating any training run, provided the JSON registry and checkpoint files are available.

### Hardware and Runtime

Experiments are executed on a Kaggle environment equipped with **two NVIDIA T4 GPUs** (16 GB VRAM each), used in data-parallel mode via the `ultralytics` multi-GPU interface.


## 2. Preliminaries Operations

### Installation

In [ ]:
!pip install -q ultralytics opencv-python-headless scipy
!pip install -q pandas tqdm

### Imports

In [ ]:
import os
import cv2
import json
import shutil
import random
from pathlib import Path
import io
import contextlib

import numpy as np
import scipy.io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
from tqdm import tqdm

import torch
import yaml
from ultralytics import YOLO
from ultralytics.utils.torch_utils import model_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPUs available: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

### Centralised Configuration

All hyperparameters, dataset paths, model variants, and output directories are defined in a single `Config` class. This centralisation ensures that no magic numbers or hardcoded strings appear in the rest of the notebook: every subsequent cell reads exclusively from the `cfg` instance. After instantiation, output directories are created, all random seeds are fixed for reproducibility, and the active device string is resolved to either a comma-separated multi-GPU identifier or a single-device fallback.

In [ ]:
class Config:
    #Dataset Paths
    prw_root      = '/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild'
    prw_yolo_root = '/kaggle/working/PRW_YOLO'

    #CrowdHuman
    ch_img_train  = '/kaggle/input/datasets/leducnhuan/crowdhuman/CrowdHuman/Images'
    ch_img_val    = '/kaggle/input/datasets/leducnhuan/crowdhuman/CrowdHuman/Images_val'
    ch_odgt_train = '/kaggle/input/datasets/leducnhuan/crowdhuman/CrowdHuman/annotation_train.odgt'
    ch_odgt_val   = '/kaggle/input/datasets/leducnhuan/crowdhuman/CrowdHuman/annotation_val.odgt'
    ch_yolo_root  = '/kaggle/working/CrowdHuman_YOLO'

    #Models
    ablation_models = {
        'small':  {'weights': 'yolo26s.pt', 'freeze_backbone': 10, 'batch': 64,  'display': 'YOLO26-Small'},
        'large':  {'weights': 'yolo26l.pt', 'freeze_backbone': 10, 'batch': 24,  'display': 'YOLO26-Large'},
    }

    #Multi-GPU (Kaggle dual T4)
    use_multi_gpu = True
    device_ids    = [0, 1]

    #CrowdHuman intermediate training
    ch_epochs    = 30
    ch_lr0       = 5e-4
    ch_lrf       = 0.01
    ch_warmup_ep = 3.0
    ch_patience  = 10

    #PRW Fine-tuning - Full
    full_epochs    = 20
    full_lr0       = 1e-4
    full_lrf       = 0.01
    full_warmup_ep = 5.0
    full_patience  = 15

    #PRW Fine-tuning - Partial (freeze backbone 0-9)
    partial_epochs    = 15
    partial_lr0       = 3e-4
    partial_lrf       = 0.01
    partial_warmup_ep = 5.0
    partial_patience  = 15

    #Common training settings
    img_size        = 640
    optimizer       = 'AdamW'
    momentum        = 0.937
    weight_decay    = 5e-4
    warmup_momentum = 0.8
    warmup_bias_lr  = 0.1
    box_loss_gain   = 7.5
    cls_loss_gain   = 0.5
    dfl_loss_gain   = 1.5
    hsv_h = 0.015; hsv_s = 0.7; hsv_v = 0.4
    degrees = 0.0;  translate = 0.1; scale = 0.5
    shear = 0.0;    perspective = 0.0
    flipud = 0.0;   fliplr = 0.5
    mosaic = 1.0;   mixup = 0.1;  copy_paste = 0.0

    #Evaluation
    eval_conf = 0.001
    eval_iou  = 0.5

    #Profiling
    n_warmup_iters = 100
    n_timed_iters  = 500

    #Output paths
    project     = '/kaggle/working/yolo_runs'
    results_dir = Path('/kaggle/working/yolo_ablation')
    plots_dir   = results_dir / 'plots'
    seed        = 42
    workers     = 8
    save_period = 10

#Instantiate config and create output directories
cfg = Config()
cfg.results_dir.mkdir(parents=True, exist_ok=True)
cfg.plots_dir.mkdir(parents=True, exist_ok=True)

#Global results registry, populated incrementally throughout the notebook
RESULTS      = {}
RESULTS_PATH = cfg.results_dir / 'all_results.json'

#Fix all random seeds for full reproducibility
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

#Resolve device string: "0,1" for dual-GPU, single device id otherwise
_device_str = (','.join(map(str, cfg.device_ids))
               if cfg.use_multi_gpu and torch.cuda.device_count() > 1
               else str(device))

print('Configuration loaded.')
print(f'Multi-GPU device string : {_device_str}')
print(f'Results dir             : {cfg.results_dir}')

### Results Registry and Persistence

The three functions below manage the serialisation and deserialisation of the global `RESULTS` dictionary, which accumulates evaluation metrics and profiling data for every run throughout the notebook.

- `_json_safe(v)`: a recursive helper that converts NumPy scalar types (`np.floating`, `np.integer`) to native Python types before serialisation, since the standard `json` module does not support NumPy types natively. Dictionaries and lists are traversed recursively; all other types are returned unchanged.
- `save_results()`: serialises the current state of `RESULTS` to disk as a formatted JSON file. It is called after every evaluation so that results are preserved even in case of kernel interruption.
- `load_results()`: reloads `RESULTS` from the JSON file on disk, restoring the full registry without re-running any experiment. It is primarily intended for use after a kernel restart.


In [ ]:
def _json_safe(v):
    """Recursively convert numpy scalars to Python native types."""
    if isinstance(v, (float, np.floating)): return round(float(v), 6)
    if isinstance(v, (int,   np.integer)):  return int(v)
    if isinstance(v, dict):  return {kk: _json_safe(vv) for kk, vv in v.items()}
    if isinstance(v, list):  return [_json_safe(x) for x in v]
    return v

def save_results():
    """Persist RESULTS dict to disk."""
    with open(RESULTS_PATH, 'w') as f:
        json.dump(_json_safe(RESULTS), f, indent=2)
    print(f'Results saved in {RESULTS_PATH}')

def load_results():
    """Reload RESULTS from disk (useful after kernel restart)."""
    global RESULTS
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            RESULTS = json.load(f)
        print(f'Loaded {len(RESULTS)} runs from {RESULTS_PATH}')
    else:
        print('No saved results found')

## 3. Dataset Utilities

### PRW to YOLO Conversion
This cell defines the utilities needed to convert the PRW dataset into the directory structure and annotation format expected by the `ultralytics` training pipeline.

* `convert_prw_to_yolo` iterates over the train and test splits, reading the per-split frame lists from the original `.mat` index files. For each frame, it copies the image into the standard `images/{split}/` directory and converts the bounding-box annotations from absolute pixel coordinates `(x, y, w, h)` to YOLO normalised format `(class x_c y_c w_n h_n)`, clipping all values to `[0, 1]` to handle any out-of-bounds annotations. Each label is written to a corresponding `.txt` file under `labels/{split}/`. Images with no valid boxes are discarded. The PRW test split is mapped to the YOLO `val` split, as is standard practice.
* `make_yaml` generates the `data.yaml` descriptor file required by `ultralytics` to locate the dataset and define the single class (`person`).

The conversion is executed only once: if the output directory already exists, the cell skips the conversion and reports the number of images already available in each split.

In [ ]:
def convert_prw_to_yolo(prw_root, output_root):
    """Convert PRW dataset annotations to YOLO format (class xc yc w h, normalised)."""
    os.makedirs(output_root, exist_ok=True)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(output_root, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_root, 'labels', split), exist_ok=True)

    frames_dir = os.path.join(prw_root, 'frames')
    ann_dir    = os.path.join(prw_root, 'annotations')
    stats      = {'train': {'images': 0, 'boxes': 0}, 'val': {'images': 0, 'boxes': 0}}

    for mode, yolo_split in [('train', 'train'), ('test', 'val')]:
        split_file = os.path.join(prw_root, f'frame_{mode}.mat')
        if not os.path.exists(split_file):
            print(f'Warning: {split_file} not found'); continue

        mat = scipy.io.loadmat(split_file)
        key = [k for k in mat.keys() if not k.startswith('__')][0]
        raw = mat[key].flatten()

        ann_files = []
        for item in raw:
            name     = str(item[0]) if isinstance(item, np.ndarray) else str(item)
            ann_name = name + '.jpg.mat'
            if os.path.exists(os.path.join(ann_dir, ann_name)):
                ann_files.append(ann_name)

        print(f'Processing {mode} split ({len(ann_files)} images)…')
        for ann_file in tqdm(ann_files, desc=mode):
            img_name = ann_file.replace('.mat', '').replace('.jpg', '') + '.jpg'
            img_path = os.path.join(frames_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            H, W = img.shape[:2]

            shutil.copy(img_path, os.path.join(output_root, 'images', yolo_split, img_name))

            boxes_raw   = scipy.io.loadmat(os.path.join(ann_dir, ann_file)).get('box_new', [])
            label_lines = []
            for box in boxes_raw:
                x, y, w, h = float(box[1]), float(box[2]), float(box[3]), float(box[4])
                if w <= 0 or h <= 0: continue
                xc = np.clip((x + w/2) / W, 0, 1)
                yc = np.clip((y + h/2) / H, 0, 1)
                wn = np.clip(w / W, 0, 1)
                hn = np.clip(h / H, 0, 1)
                label_lines.append(f'0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n')
                stats[yolo_split]['boxes'] += 1

            if label_lines:
                lbl_name = img_name.replace('.jpg', '.txt')
                with open(os.path.join(output_root, 'labels', yolo_split, lbl_name), 'w') as f:
                    f.writelines(label_lines)
                stats[yolo_split]['images'] += 1

    print('PRW conversion done.')
    print(f"  train : {stats['train']['images']} images, {stats['train']['boxes']} boxes")
    print(f"  val   : {stats['val']['images']} images, {stats['val']['boxes']} boxes")
    return stats


def make_yaml(root, out_path, nc=1, names=None):
    """Write a data.yaml file for YOLO training."""
    names = names or ['person']
    with open(out_path, 'w') as f:
        yaml.dump({'path': str(root), 'train': 'images/train',
                   'val': 'images/val', 'nc': nc, 'names': names},
                  f, default_flow_style=False)
    print(f'YAML saved in {out_path}')
    return out_path


# Convert PRW once
if not os.path.exists(cfg.prw_yolo_root):
    convert_prw_to_yolo(cfg.prw_root, cfg.prw_yolo_root)
else:
    n_tr = len(os.listdir(os.path.join(cfg.prw_yolo_root, 'images', 'train')))
    n_va = len(os.listdir(os.path.join(cfg.prw_yolo_root, 'images', 'val')))
    print(f'PRW YOLO dataset already exists - train: {n_tr}, val: {n_va}')

prw_yaml = make_yaml(cfg.prw_yolo_root, os.path.join(cfg.prw_yolo_root, 'data.yaml'))

### CrowdHuman to YOLO Conversion

This cell defines the utilities needed to convert the CrowdHuman dataset into the YOLO-compatible format, following the same directory structure used for PRW.

* `_odgt_to_yolo` processes a single `.odgt` annotation file, where each line is a JSON record containing the image ID and a list of ground-truth boxes. For each record, the function locates the source image (trying multiple extensions), reads its dimensions, and iterates over the annotated instances. Only `person`-tagged boxes that are not flagged as ignored are retained. Full-body boxes (`fbox`, in absolute pixel coordinates `(x, y, w, h)`) are converted to YOLO normalised format and clipped to `[0, 1]`. Images with no valid boxes after filtering are discarded. Each retained image is physically copied into the destination `images/{split}/` directory: this step is required because the `ultralytics` training pipeline locates label files by replacing the `images/` component of the image path with `labels/`, which only works correctly if images reside inside the YOLO root tree rather than being accessed via symlinks to an external location.
* `convert_crowdhuman_to_yolo` orchestrates the conversion for both splits by calling `_odgt_to_yolo` on each, checking beforehand whether the output directories are already populated to avoid redundant processing. On completion it calls `make_yaml` to write the dataset descriptor. The CrowdHuman paths on `cfg` are updated inline before calling the function, to match the layout of the `leducnhuan/crowdhuman` Kaggle dataset.


In [ ]:
def _odgt_to_yolo(odgt_path, img_src_dir, img_dst_dir, label_dir, split_name=''):
    os.makedirs(img_dst_dir, exist_ok=True)
    os.makedirs(label_dir,   exist_ok=True)
    converted, total_boxes, skipped = 0, 0, 0

    with open(odgt_path) as f:
        lines = f.readlines()

    for line in tqdm(lines, desc=f'CrowdHuman {split_name}'):
        rec    = json.loads(line.strip())
        img_id = rec['ID']

        # Locate source image
        img_src = None
        for ext in ['.jpg', '.JPG', '.jpeg', '.png']:
            candidate = os.path.join(img_src_dir, img_id + ext)
            if os.path.exists(candidate):
                img_src = candidate
                break
        if img_src is None:
            skipped += 1
            continue

        img = cv2.imread(img_src)
        if img is None:
            skipped += 1
            continue
        H, W = img.shape[:2]

        # Build label lines
        label_lines = []
        for ann in rec.get('gtboxes', []):
            if ann.get('tag') != 'person':
                continue
            if ann.get('extra', {}).get('ignore', 0) == 1:
                continue
            fbox = ann.get('fbox')      # [x, y, w, h] pixel coords
            if fbox is None:
                continue
            x, y, w, h = fbox
            if w <= 0 or h <= 0:
                continue
            xc = np.clip((x + w / 2) / W, 0, 1)
            yc = np.clip((y + h / 2) / H, 0, 1)
            wn = np.clip(w / W, 0, 1)
            hn = np.clip(h / H, 0, 1)
            label_lines.append(f'0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n')
            total_boxes += 1

        if not label_lines:
            continue

        # Copy image to dst (needed so YOLO path substitution works)
        img_dst = os.path.join(img_dst_dir, os.path.basename(img_src))
        if not os.path.exists(img_dst):
            shutil.copy2(img_src, img_dst)

        # Write label file
        lbl_path = os.path.join(label_dir, img_id + '.txt')
        with open(lbl_path, 'w') as f:
            f.writelines(label_lines)
        converted += 1

    print(f'  {split_name}: {converted} images, {total_boxes} boxes '
          f'({skipped} skipped)')
    return converted


def convert_crowdhuman_to_yolo(cfg):
    """
    Convert leducnhuan/crowdhuman (Kaggle) to YOLO format under cfg.ch_yolo_root.
    """
    for split, img_src, odgt in [
            ('train', cfg.ch_img_train, cfg.ch_odgt_train),
            ('val',   cfg.ch_img_val,   cfg.ch_odgt_val)]:

        img_dst   = os.path.join(cfg.ch_yolo_root, 'images', split)
        label_dir = os.path.join(cfg.ch_yolo_root, 'labels', split)

        if not os.path.exists(odgt):
            print(f'WARNING: {odgt} not found - skipping {split}.')
            continue
        if not os.path.exists(img_src):
            print(f'WARNING: {img_src} not found - skipping {split}.')
            continue

        # Check if already converted (both images and labels present)
        n_imgs   = len(os.listdir(img_dst))   if os.path.exists(img_dst)   else 0
        n_labels = len(os.listdir(label_dir)) if os.path.exists(label_dir) else 0
        if n_imgs > 0 and n_labels > 0:
            print(f'CrowdHuman {split} already converted '
                  f'({n_imgs} images, {n_labels} labels) - skipping.')
            continue

        print(f'Converting CrowdHuman {split}…')
        _odgt_to_yolo(odgt, img_src, img_dst, label_dir, split_name=split)

    ch_yaml_path = os.path.join(cfg.ch_yolo_root, 'data.yaml')
    ch_yaml = make_yaml(cfg.ch_yolo_root, ch_yaml_path)
    return ch_yaml

ch_yaml = convert_crowdhuman_to_yolo(cfg)

## 4. Training, Evaluation and Visualization Pipeline

### Model and Training Utilities

This section defines the core functions used to load models and execute training runs uniformly across all experimental phases.

* `load_model` is a thin wrapper around the `ultralytics` YOLO constructor, accepting either a filename (e.g. `yolo26s.pt`) or a full path to a checkpoint.
* `_train_kwargs` assembles the complete keyword-argument dictionary for `model.train()`, reading all augmentation, optimiser, and infrastructure settings from `cfg`. Separating shared arguments into this helper avoids repetition across phases and guarantees that every run uses an identical base configuration. An optional `extra` dictionary can be passed to override or extend specific keys.
* `run_training` is the unified training entry point used by all phases. It loads the model, merges the shared kwargs with the run-specific parameters (dataset, epochs, batch size, learning rate schedule, and optionally the number of frozen backbone layers), and launches training. The `freeze` parameter maps directly to the `ultralytics` `freeze` argument: when set to an integer `n`, the first `n` backbone layers are kept frozen during the run, implementing the partial fine-tuning strategy evaluated in Phase 2.
* `best_weights_path` is a convenience helper that returns the canonical path to the `best.pt` checkpoint for a given run name, following the standard `ultralytics` output directory structure.

In [ ]:
def load_model(weights_path):
    """Load a YOLO model from a weights path or model-name string."""
    return YOLO(weights_path)


def _train_kwargs(extra=None):
    """Common augmentation and optimizer kwargs shared across all runs."""
    base = dict(
        optimizer       = cfg.optimizer,
        momentum        = cfg.momentum,
        weight_decay    = cfg.weight_decay,
        warmup_momentum = cfg.warmup_momentum,
        warmup_bias_lr  = cfg.warmup_bias_lr,
        box             = cfg.box_loss_gain,
        cls             = cfg.cls_loss_gain,
        dfl             = cfg.dfl_loss_gain,
        hsv_h=cfg.hsv_h, hsv_s=cfg.hsv_s, hsv_v=cfg.hsv_v,
        degrees=cfg.degrees, translate=cfg.translate, scale=cfg.scale,
        shear=cfg.shear, perspective=cfg.perspective,
        flipud=cfg.flipud, fliplr=cfg.fliplr,
        mosaic=cfg.mosaic, mixup=cfg.mixup, copy_paste=cfg.copy_paste,
        seed=cfg.seed, workers=cfg.workers,
        project=cfg.project,
        plots=True, val=True, verbose=True,
        device=_device_str,
    )
    if extra:
        base.update(extra)
    return base


def run_training(weights, data_yaml, name, epochs, batch,
                 lr0, lrf, warmup_epochs, patience, freeze=None):
    """
    Unified training wrapper.
    freeze: int | None - number of backbone layers to freeze (Ultralytics param).
    """
    model = load_model(weights)
    kw = _train_kwargs()
    kw.update(dict(
        data          = data_yaml,
        epochs        = epochs,
        batch         = batch,
        imgsz         = cfg.img_size,
        lr0           = lr0,
        lrf           = lrf,
        warmup_epochs = warmup_epochs,
        patience      = patience,
        name          = name,
        exist_ok      = True,
        save_period   = cfg.save_period,
    ))
    if freeze is not None:
        kw['freeze'] = freeze
    return model.train(**kw)


def best_weights_path(name):
    """Return the path to best.pt for a given run name."""
    return os.path.join(cfg.project, name, 'weights', 'best.pt')

### Evaluation and Profiling Utilities

This section defines the functions responsible for quantifying both the detection quality and the computational efficiency of each model checkpoint.

* `evaluate_model` runs the standard `ultralytics` validation loop on the PRW validation split, using the confidence and IoU thresholds defined in `cfg`. It returns a flat dictionary with four detection metrics (`mAP@0.5`, `mAP@0.5:0.95`, `Precision`, `Recall`) and two speed metrics (`inference_ms`, `throughput`) extracted directly from `res.speed`, which ultralytics computes internally as the average over the entire validation set with real batches.
* `profile_model` reads parameter count and GFLOPs from `ultralytics.utils.torch_utils.model_info`, the same function that produces the model summary line printed at the start of each run. This guarantees that the values stored in `RESULTS` are exactly consistent with the ultralytics-reported figures. `model_info` is called with `verbose=True` (required to obtain return values in this ultralytics version) but with stdout suppressed via `contextlib.redirect_stdout` to avoid polluting the notebook output. Importantly, this function must be called **before** `evaluate_model` on the same model instance: `val()` internally fuses Conv and BatchNorm layers for inference efficiency, which would alter the parameter count and GFLOPs if read afterwards.
* `full_eval_and_profile` is the single entry point called at the end of each experimental phase. It loads the model once and shares the same instance across both `profile_model` and `evaluate_model`, avoiding the double-loading issue that would otherwise produce two summary lines in the output. Results are merged into a single record, annotated with metadata, stored in the global `RESULTS` registry, and immediately persisted to disk. If the key is already present, the function skips computation entirely, making it safe to re-run cells after a kernel interruption.



In [ ]:
def evaluate_model(model, data_yaml):
    """Run YOLO .val() and return detection metrics and speed from res.speed."""
    res = model.val(
        data=data_yaml, split='val', batch=32, imgsz=cfg.img_size,
        conf=cfg.eval_conf, iou=cfg.eval_iou, device=_device_str,
        save_json=False, plots=False, verbose=False,
    )
    inference_ms = float(res.speed['inference'])
    return {
        'mAP50':        float(res.box.map50),
        'mAP50_95':     float(res.box.map),
        'precision':    float(res.box.mp),
        'recall':       float(res.box.mr),
        'inference_ms': inference_ms,
        'throughput':   round(1000 / inference_ms, 1) if inference_ms > 0 else 0.0,
    }


@torch.no_grad()
def profile_model(model, img_size=640):
    """Measure params and GFLOPs using ultralytics model_info().
    Must be called before evaluate_model() to avoid reading fused values."""

    with contextlib.redirect_stdout(io.StringIO()):
        nlayers, nparams, ngrads, flops_giga = model_info(model.model, verbose=True, imgsz=img_size)

    trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

    return {
        'total_params':     nparams,
        'trainable_params': trainable_params,
        'flops_giga':       flops_giga,
    }


def full_eval_and_profile(model_or_path, data_yaml, run_key,
                          display_name='', strategy='', note=''):
    """Evaluate + profile a model, store in RESULTS[run_key], persist. Skips if already done."""
    if run_key in RESULTS:
        print(f'[SKIP] {run_key} already in RESULTS.')
        return RESULTS[run_key]

    print(f'\n{"="*65}')
    print(f'  Evaluating : {run_key}  ({display_name})')
    print(f'{"="*65}')

    model = model_or_path if isinstance(model_or_path, YOLO) else YOLO(model_or_path)

    hw      = profile_model(model)
    metrics = evaluate_model(model, data_yaml)

    entry = {**hw, **metrics,
             'display_name': display_name, 'strategy': strategy, 'note': note}
    RESULTS[run_key] = entry
    save_results()

    print(f"  mAP@50={metrics['mAP50']:.4f}  mAP@50-95={metrics['mAP50_95']:.4f}")
    print(f"  Params: {hw['total_params']/1e6:.1f}M  FLOPs: {hw['flops_giga']:.1f}G")
    print(f"  Latency: {metrics['inference_ms']:.1f} ms  Throughput: {metrics['throughput']:.0f} img/s")

### Plotting Utilities

This section defines all visualisation functions used throughout the notebook. A shared style (`seaborn-v0_8-darkgrid`) and a fixed ten-colour palette are set globally so that all figures maintain a consistent appearance.

* `plot_training_curves` reads the `results.csv` file produced by `ultralytics` at the end of each training run and generates a six-panel figure: total training loss, the three individual loss components (box, classification, and DFL), precision and recall curves, and mAP curves. It is called once per training phase to inspect convergence behaviour and detect overfitting or underfitting.
* `_radar_chart` is a private helper that produces a four-axis polar chart comparing `mAP@0.5`, `mAP@0.5:0.95`, `Precision`, and `Recall` across multiple runs simultaneously. Each model is represented by a coloured polygon, with a semi-transparent fill to aid visual comparison. It is not called directly but is used internally by `plot_comparison`.
* `plot_comparison` is the standard per-phase comparison function. It produces two figures for a given subset of results: a grouped bar chart showing `mAP@0.5` and `Recall` side by side for each run, and the radar chart described above. The two metrics were selected to give a direct picture of detection quality from the perspective of the downstream Re-ID module, where recall is the primary concern. It is called at the end of each phase to summarise the relative performance of the configurations evaluated in that phase.
* `plot_final_comparison` generates the summary figure for the entire ablation. It produces a four-panel bar chart covering detection quality (`mAP@0.5`), model size (parameters in millions), compute cost (GFLOPs), and inference speed (latency in ms per image), providing a complete quality-vs-efficiency overview across all runs.


In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
rcParams.update({'font.size': 10, 'axes.titlesize': 12, 'figure.titlesize': 13})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E',
           '#BC4B51', '#5F4B8B', '#E08D79', '#1B998B', '#E84855']


def plot_training_curves(results_csv_path, title_prefix='', save_path=None):
    """Plot YOLO training loss + metric curves from results.csv."""
    if not os.path.exists(results_csv_path):
        print(f'results.csv not found at {results_csv_path}'); return
    df = pd.read_csv(results_csv_path)
    df.columns = df.columns.str.strip()

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(f'{title_prefix}Training Curves', fontweight='bold')

    ax = plt.subplot(2, 3, 1)
    if all(c in df.columns for c in ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']):
        total = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
        ax.plot(df['epoch'], total, 'b-o', lw=2, ms=4)
    ax.set(xlabel='Epoch', ylabel='Loss', title='Total Train Loss'); ax.grid(True, alpha=.3)

    for ax_idx, (col, lbl) in enumerate([
            ('train/box_loss', 'Box Loss'),
            ('train/cls_loss', 'Cls Loss'),
            ('train/dfl_loss', 'DFL Loss')], start=2):
        ax = plt.subplot(2, 3, ax_idx)
        if col in df.columns:
            ax.plot(df['epoch'], df[col], 'b-', lw=1.5, alpha=.8)
        ax.set(xlabel='Epoch', ylabel='Loss', title=lbl); ax.grid(True, alpha=.3)

    ax = plt.subplot(2, 3, 5)
    for col, lbl, c in [('metrics/precision(B)', 'Precision', 'g'),
                         ('metrics/recall(B)',    'Recall',    'm')]:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], f'{c}-', lw=2, label=lbl)
    ax.set(xlabel='Epoch', ylabel='Value', title='Precision & Recall')
    ax.legend(); ax.grid(True, alpha=.3)

    ax = plt.subplot(2, 3, 6)
    for col, lbl, c in [('metrics/mAP50(B)',    'mAP@0.5',      '#2E86AB'),
                         ('metrics/mAP50-95(B)', 'mAP@0.5:0.95', '#F18F01')]:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], '-', color=c, lw=2, label=lbl)
    ax.set(xlabel='Epoch', ylabel='mAP', title='Mean Average Precision')
    ax.legend(); ax.grid(True, alpha=.3)

    plt.tight_layout()
    sp = save_path or str(cfg.plots_dir / f'{title_prefix.strip().replace(" ", "_")}_curves.png')
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    plt.show()


def _radar_chart(results_dict, title, save_path=None):
    """4-axis radar: mAP@50, mAP@50-95, Precision, Recall."""
    metrics     = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']
    metric_keys = ['mAP50',  'mAP50_95',  'precision', 'recall']
    N      = len(metrics)
    angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=7, color='grey')

    for (name, v), color in zip(results_dict.items(), PALETTE):
        vals = [v.get(mk, 0) * 100 for mk in metric_keys]
        vals += vals[:1]
        ax.plot(angles, vals, '-o', lw=2, color=color, ms=6,
                label=v.get('display_name', name))
        ax.fill(angles, vals, alpha=0.08, color=color)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper left')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_comparison(results_dict, title='Comparison', save_dir=None):
    """Two-panel: grouped bar (mAP@50 and Recall), radar."""
    save_dir = save_dir or str(cfg.plots_dir)
    names   = list(results_dict.keys())
    map50   = np.array([results_dict[n].get('mAP50',   0) * 100 for n in names])
    recall  = np.array([results_dict[n].get('recall',  0) * 100 for n in names])

    # 1. Grouped bar
    fig, ax = plt.subplots(figsize=(max(10, len(names) * 1.6), 6))
    x, w = np.arange(len(names)), 0.35
    b1 = ax.bar(x - w/2, map50,  w, label='mAP@50',  color='#2E86AB', alpha=.85, edgecolor='k', lw=.5)
    b2 = ax.bar(x + w/2, recall, w, label='Recall',   color='#A23B72', alpha=.85, edgecolor='k', lw=.5)
    for b in list(b1) + list(b2):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + .3,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=7.5)
    ax.set(xticks=x,
           xticklabels=[results_dict[n].get('display_name', n) for n in names],
           ylabel='Score (%)', title=f'{title} - mAP@50 & Recall')
    plt.xticks(rotation=30, ha='right'); ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{title.replace(" ","_")}_bar.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # 2. Radar
    _radar_chart(results_dict, title=f'{title} - Radar',
                 save_path=os.path.join(save_dir, f'{title.replace(" ","_")}_radar.png'))


def plot_final_comparison(results_dict, save_dir=None):
    """4-panel final figure: quality / size / compute / speed."""
    save_dir = save_dir or str(cfg.plots_dir)
    names    = list(results_dict.keys())
    labels   = [results_dict[n].get('display_name', n) for n in names]
    map50    = np.array([results_dict[n].get('mAP50',        0)*100 for n in names])
    params   = np.array([results_dict[n].get('total_params', 0)/1e6 for n in names])
    flops    = np.array([results_dict[n].get('flops_giga',   0)     for n in names])
    latency  = np.array([results_dict[n].get('inference_ms', 0)     for n in names])
    colors   = PALETTE[:len(names)]
    x        = np.arange(len(names))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Final Model Comparison', fontweight='bold')

    for ax, vals, ylabel, title_ in zip(
            axes.flat,
            [map50,        params,       flops,         latency],
            ['mAP@50 (%)', 'Params (M)', 'GFLOPs',      'Latency (ms/img)'],
            ['Detection Quality', 'Model Size', 'Compute Cost', 'Inference Speed']):
        bars = ax.bar(x, vals, color=colors, alpha=.85, edgecolor='k', lw=.5)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + .01*(vals.max()+1e-9),
                    f'{v:.1f}', ha='center', va='bottom', fontsize=8)
        ax.set(xticks=x, xticklabels=labels, ylabel=ylabel, title=title_)
        plt.sca(ax); plt.xticks(rotation=30, ha='right')
        ax.grid(True, alpha=.3, axis='y')

    plt.tight_layout()
    sp = os.path.join(save_dir, 'final_comparison.png')
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    print(f'Saved in {sp}')
    plt.show()

## 5. Ablation Phases

### Phase 0 - COCO Baseline

This phase establishes the zero-shot detection baseline. YOLO26-Small is loaded directly from its official COCO pre-trained weights (`yolo26s.pt`) and evaluated on the PRW validation split without any domain-specific fine-tuning. No training is performed. The resulting metrics represent the upper bound of what the model can achieve by relying solely on features learned from COCO, and serve as the reference point against which all subsequent fine-tuning experiments are measured.


In [ ]:
model_s_coco = load_model(cfg.ablation_models['small']['weights'])

full_eval_and_profile(
    model_s_coco,
    data_yaml    = prw_yaml,
    run_key      = 'small_coco_zeroshot',
    display_name = 'Small - COCO (zero-shot)',
    strategy     = 'zero-shot',
    note         = 'COCO pretrained, no PRW fine-tuning',
)

### Phase 1 - CrowdHuman Intermediate Training

This phase investigates whether an intermediate pre-training step on CrowdHuman improves transferability to PRW before any target-domain fine-tuning. YOLO26-Small is fine-tuned on CrowdHuman for up to 30 epochs starting from the COCO weights, with early stopping controlled by `cfg.ch_patience`.

After training, both the COCO baseline (`small_coco_zeroshot`) and the CrowdHuman-pretrained model (`small_ch_zeroshot`) are evaluated zero-shot on PRW and compared. The model achieving the higher `mAP@0.5` is automatically selected as the starting point for Phase 2, stored in `best_start_weights`. This data-driven selection ensures that the fine-tuning strategy comparison in Phase 2 always begins from the strongest available initialisation.

In [ ]:
if not os.path.exists(best_weights_path('small_ch_pretrain')):
    run_training(
        weights       = cfg.ablation_models['small']['weights'],
        data_yaml     = ch_yaml,
        name          = 'small_ch_pretrain',
        epochs        = cfg.ch_epochs,
        batch         = cfg.ablation_models['small']['batch'],
        lr0           = cfg.ch_lr0,
        lrf           = cfg.ch_lrf,
        warmup_epochs = cfg.ch_warmup_ep,
        patience      = cfg.ch_patience,
    )
    print('CrowdHuman training done.')
else:
    print('CrowdHuman checkpoint already exists - skipping.')

In [ ]:
plot_training_curves(
    os.path.join(cfg.project, 'small_ch_pretrain', 'results.csv'),
    title_prefix='Phase 1 - CrowdHuman ',
)

In [ ]:
full_eval_and_profile(
    model_or_path = best_weights_path('small_ch_pretrain'),
    data_yaml     = prw_yaml,
    run_key       = 'small_ch_zeroshot',
    display_name  = 'Small - CH pretrain (zero-shot PRW)',
    strategy      = 'zero-shot',
    note          = 'COCO then CrowdHuman, zero-shot on PRW',
)

In [ ]:
phase1_base = {k: RESULTS[k] for k in ['small_coco_zeroshot', 'small_ch_zeroshot']
               if k in RESULTS}
plot_comparison(phase1_base, title='Phase 1 - Baseline Weights Comparison')

best_start_key = max(phase1_base, key=lambda k: phase1_base[k].get('mAP50', 0))
best_start_weights = (cfg.ablation_models['small']['weights']
                      if best_start_key == 'small_coco_zeroshot'
                      else best_weights_path('small_ch_pretrain'))
print(f'Best starting weights: {best_start_key}')

### Phase 2 - Fine-tuning Strategy Comparison

This phase compares two fine-tuning strategies applied to YOLO26-Small on PRW, starting from the best initialisation identified in Phase 1.

- **Full fine-tuning (`full_ft`)**: all layers are updated during training, with a conservative initial learning rate (`cfg.full_lr0 = 1e-4`) to avoid overwriting the pretrained feature representations.
- **Partial fine-tuning (`partial_ft`)**: backbone layers 0-9 are frozen and only the neck and detection head are updated, with a higher initial learning rate (`cfg.partial_lr0 = 3e-4`) since fewer parameters are being optimised. The number of frozen layers is controlled by `cfg.ablation_models['small']['freeze_backbone']`.

Both runs share the same dataset, augmentation pipeline, and early stopping patience, differing only in the `freeze` parameter passed to `run_training`. As in Phase 1, training is skipped if a valid checkpoint already exists. After evaluation, the strategy achieving the higher `mAP@0.5` is automatically selected as `best_strat_key` and propagated to Phase 3, where it is applied to the larger model scale.


In [ ]:
STRATEGY = {
    'full': {
        'freeze':  None,
        'lr0':     cfg.full_lr0,
        'lrf':     cfg.full_lrf,
        'epochs':  cfg.full_epochs,
        'patience':cfg.full_patience,
        'warmup':  cfg.full_warmup_ep,
        'label':   'Full FT',
    },
    'partial': {
        'freeze':  cfg.ablation_models['small']['freeze_backbone'],  # 10
        'lr0':     cfg.partial_lr0,
        'lrf':     cfg.partial_lrf,
        'epochs':  cfg.partial_epochs,
        'patience':cfg.partial_patience,
        'warmup':  cfg.partial_warmup_ep,
        'label':   'Partial FT (freeze bb 0–9)',
    },
}

In [ ]:
for strat_key, sc in STRATEGY.items():
    run_name = f'small_{strat_key}_ft'
    if not os.path.exists(best_weights_path(run_name)):
        print(f'\nTraining {run_name} …')
        run_training(
            weights       = best_start_weights,
            data_yaml     = prw_yaml,
            name          = run_name,
            epochs        = sc['epochs'],
            batch         = cfg.ablation_models['small']['batch'],
            lr0           = sc['lr0'],
            lrf           = sc['lrf'],
            warmup_epochs = sc['warmup'],
            patience      = sc['patience'],
            freeze        = sc['freeze'],
        )
    else:
        print(f'Checkpoint {run_name} already exists - skipping.')

In [ ]:
for strat_key, sc in STRATEGY.items():
    plot_training_curves(
        os.path.join(cfg.project, f'small_{strat_key}_ft', 'results.csv'),
        title_prefix=f'Phase 2 - {sc["label"]} ',
    )

In [ ]:
for strat_key, sc in STRATEGY.items():
    run_name = f'small_{strat_key}_ft'
    full_eval_and_profile(
        model_or_path = best_weights_path(run_name),
        data_yaml     = prw_yaml,
        run_key       = run_name,
        display_name  = f'Small - {sc["label"]}',
        strategy      = strat_key,
        note          = f'Starting from {best_start_key}',
    )

In [ ]:
phase2_results = {k: RESULTS[k] for k in [f'small_{s}_ft' for s in STRATEGY] if k in RESULTS}
plot_comparison(phase2_results, title='Phase 2 - Strategy Comparison')

best_strategy  = max(phase2_results, key=lambda k: phase2_results[k].get('mAP50', 0))
best_strat_key = best_strategy.replace('small_', '').replace('_ft', '')
print(f'Phase 2 winner : {best_strategy}  (mAP@50 = {phase2_results[best_strategy]["mAP50"]*100:.2f}%)')

### Phase 3 - Scale-up: YOLO26-Large on PRW

This phase applies the best fine-tuning strategy identified in Phase 2 to YOLO26-Large, in order to assess whether the performance gains obtained on the Small scale generalise to a larger model with higher capacity.

The initialisation strategy for the Large model mirrors the outcome of Phase 1: if the CrowdHuman pre-training was found to be beneficial for the Small model (`best_start_key == 'small_ch_zeroshot'`), the Large model is first pre-trained on CrowdHuman using the same hyperparameters (`cfg.ch_epochs`, `cfg.ch_lr0`) before fine-tuning on PRW. Otherwise, training starts directly from the official COCO weights (`yolo26l.pt`). This conditional logic ensures that the training pipeline applied to the Large model is strictly consistent with the one that produced the best results at the Small scale, making the cross-scale comparison methodologically sound.

In [ ]:
sc_l       = STRATEGY[best_strat_key]
run_name_l = f'large_{best_strat_key}_ft'

if best_start_key == 'small_ch_zeroshot':
    if not os.path.exists(best_weights_path('large_ch_pretrain')):
        print('Phase 1 winner was CrowdHuman - pre-training Large on CrowdHuman first…')
        run_training(
            weights       = cfg.ablation_models['large']['weights'],
            data_yaml     = ch_yaml,
            name          = 'large_ch_pretrain',
            epochs        = cfg.ch_epochs,
            batch         = cfg.ablation_models['large']['batch'],
            lr0           = cfg.ch_lr0,
            lrf           = cfg.ch_lrf,
            warmup_epochs = cfg.ch_warmup_ep,
            patience      = cfg.ch_patience,
        )
    else:
        print('Large CrowdHuman checkpoint already exists - skipping.')
    large_start_weights = best_weights_path('large_ch_pretrain')
else:
    large_start_weights = cfg.ablation_models['large']['weights']

print(f'Large starting weights: {large_start_weights}')

if not os.path.exists(best_weights_path(run_name_l)):
    run_training(
        weights       = large_start_weights,
        data_yaml     = prw_yaml,
        name          = run_name_l,
        epochs        = sc_l['epochs'],
        batch         = cfg.ablation_models['large']['batch'],
        lr0           = sc_l['lr0'],
        lrf           = sc_l['lrf'],
        warmup_epochs = sc_l['warmup'],
        patience      = sc_l['patience'],
        freeze        = sc_l['freeze'],
    )
else:
    print(f'Checkpoint {run_name_l} already exists - skipping.')


In [ ]:
plot_training_curves(os.path.join(cfg.project, run_name_l, 'results.csv'),
                    title_prefix=f'Phase 3 - Large {sc_l["label"]} ')

In [ ]:
full_eval_and_profile(
    best_weights_path(run_name_l), prw_yaml, run_name_l,
    display_name=f'Large - {sc_l["label"]}', strategy=best_strat_key,
    note='Scale-up, COCO init',
)

## 6. Final Comparison

This section aggregates the results of all experimental phases into a unified view. All run keys are collected from the `RESULTS` registry, filtered to retain only completed experiments, and passed to `plot_comparison` and `plot_final_comparison` for visual analysis. A summary DataFrame is then constructed, sorted by `mAP@0.5` in descending order, printed to the notebook output, and exported as a CSV file to `cfg.results_dir` for external reference.

In [ ]:
all_run_keys = ['small_coco_zeroshot', 'small_ch_zeroshot',
                'small_full_ft', 'small_partial_ft',
                run_name_l]
all_results = {k: RESULTS[k] for k in all_run_keys if k in RESULTS}

plot_comparison(all_results, title='All Runs')
plot_final_comparison(all_results)

In [ ]:
rows = []
for rk, v in all_results.items():
    rows.append({
        'Run':              rk,
        'Display Name':     v.get('display_name', rk),
        'Strategy':         v.get('strategy', '-'),
        'mAP@50 (%)':       round(v.get('mAP50',        0)*100, 2),
        'mAP@50-95 (%)':    round(v.get('mAP50_95',     0)*100, 2),
        'Precision (%)':    round(v.get('precision',     0)*100, 2),
        'Recall (%)':       round(v.get('recall',        0)*100, 2),
        'Params (M)':       round(v.get('total_params',  0)/1e6, 1),
        'GFLOPs':           round(v.get('flops_giga',    0), 2),
        'Latency (ms)':     round(v.get('inference_ms',  0), 2),
        'Throughput (i/s)': round(v.get('throughput',    0), 1),
        'Note':             v.get('note', ''),
    })

df_final = pd.DataFrame(rows).set_index('Run').sort_values('mAP@50 (%)', ascending=False)
print('\n' + '='*120)
print('COMPLETE RESULTS (sorted by mAP@50)')
print('='*120)
print(df_final.to_string())

out_csv = cfg.results_dir / 'all_results.csv'
df_final.to_csv(out_csv)

## 7. Conclusions

### Summary of Results

The ablation study yields five configurations spanning zero-shot evaluation, intermediate domain pre-training, two fine-tuning strategies, and a scale-up experiment. The results are consistent with established findings in transfer learning for object detection, and each phase produces outcomes that are largely expected from a methodological standpoint, with a few noteworthy observations.

### Effect of Domain-Specific Fine-tuning

The most substantial performance gain is observed when transitioning from zero-shot evaluation to target-domain fine-tuning. The COCO zero-shot baseline (`small_coco_zeroshot`) achieves `mAP@0.5 = 85.82%` and `Recall = 79.42%`, which already represents a competitive starting point given that COCO contains a `person` class with broad visual diversity. Fine-tuning on PRW raises `mAP@0.5` to `94.91-94.96%` (+9 pp) and recall to `89.32-89.33%` (+9.9 pp), confirming that domain adaptation is essential even when the source domain partially overlaps with the target task. The recall improvement is particularly significant in the context of person search: the zero-shot model misses approximately one in five pedestrians, a rate that would propagate as unrecoverable errors to the downstream Re-ID module.

### Effect of CrowdHuman Pre-training

The intermediate CrowdHuman pre-training step produces a measurable improvement in zero-shot PRW performance (`mAP@0.5`: `85.82%` -> `88.23%`, +2.4 pp; `Recall`: `79.42%` -> `83.41%`, +4 pp), validating the hypothesis that exposure to crowded surveillance-like scenes improves feature generalisation before target-domain fine-tuning. The recall gain in particular suggests that the CrowdHuman pre-training helps the model detect partially occluded or small-scale pedestrians that the COCO-only model tends to miss, which is precisely the failure mode most harmful in a person search pipeline.

### Full vs. Partial Fine-tuning

The comparison between full fine-tuning (`mAP@0.5 = 94.96%`, `Recall = 89.32%`) and partial fine-tuning with frozen backbone layers 0-9 (`mAP@0.5 = 94.91%`, `Recall = 89.33%`) reveals a negligible difference of 0.05 pp in `mAP@0.5` and a practically identical recall. This outcome is expected for a single-class detection task on a dataset with limited visual diversity: when the target domain requires only the detection of a single category, the backbone features learned on COCO are already sufficiently general, and freezing them does not impair the model's ability to adapt. From a person search perspective, both strategies deliver equivalent recall, meaning that the partial fine-tuning approach is preferable in practice: it achieves the same detection coverage with a higher learning rate, fewer updated parameters, and reduced risk of catastrophic forgetting of pretrained representations.

### Scale-up: YOLO26-Large

Scaling from Small to Large produces the strongest result across all metrics (`mAP@0.5 = 96.24%`, `mAP@0.5:0.95 = 70.60%`, `Recall = 91.38%`). The recall improvement over the best Small configuration is +2.05 pp, meaning that the Large model recovers approximately 2 additional pedestrians out of every 100 that the Small model misses. While modest in relative terms, this gain is non-negligible in a person search context where missed detections are unrecoverable. The improvement is more pronounced on `mAP@0.5:0.95` (+2.49 pp), suggesting that the Large model not only detects more pedestrians but also localises them more precisely, which benefits the quality of the image crops passed to the Re-ID module. This gain comes at a substantial computational cost: the Large model has 2.6x more parameters (26.2M vs. 9.9M), 4.1x higher GFLOPs (93.12 vs. 22.50), and 3.6x higher latency (27.88 ms vs. 7.72 ms per image).

### Model Selection for Person Search

From the perspective of the downstream two-stage person search pipeline, the choice between Small and Large involves an explicit trade-off between recall and inference speed. The Large model is the preferred candidate when detection coverage is the primary concern and computational resources allow it, given its superior recall (91.38%) and localisation accuracy. The Small model is a viable alternative in latency-constrained deployments (7.72 ms vs. 27.88 ms per image), where its recall of 89.32% remains well above the zero-shot baseline and represents an acceptable operating point for the Re-ID stage.

### Methodological Note on Parameter Counts

A non-trivial discrepancy in parameter counts was identified during the study. The zero-shot COCO model reports 10.0M parameters under direct PyTorch counting, while all fine-tuned models report 9.9M. This difference of approximately 61,000 parameters is entirely attributable to the detection head being rebuilt by `ultralytics` when adapting the model to a single-class dataset: the final classification convolution changes from `Conv2d(128, 80)` to `Conv2d(128, 1)` across three output scales and two head branches (`cv3` and `one2one_cv3`), accounting for exactly `6 * (10320 - 129) = 61,146` fewer parameters. The profiling values reported in the table are those produced by `ultralytics model_info()` on the fused model, which are consistent across all runs and directly comparable.

## References

[1] Zheng, L., Zhang, H., Sun, S., Chandraker, M., Yang, Y., and Tian, Q.
"Person Re-identification in the Wild."
*IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*, 2017.
https://arxiv.org/abs/1604.02531

[2] Shao, S., Zhao, Z., Li, B., Xiao, T., Yu, G., Zhang, X., and Sun, J.
"CrowdHuman: A Benchmark for Detecting Human in a Crowd."
*arXiv preprint arXiv:1805.00123*, 2018.
https://arxiv.org/abs/1805.00123

[3] Ultralytics.
"YOLO26: Key Architectural Enhancements and Performance Benchmarking for Real-Time Object Detection."
*arXiv preprint arXiv:2509.25164*, 2025.
https://arxiv.org/abs/2509.25164
